# BES Scientific Discourse NLP Pipeline

Reconstructed notebook aligned with the revised manuscript. **Run only with the actual corpus.** T1–T4 are thematic domains; no maturity ground-truth is assumed.

In [ ]:
from pathlib import Path
import sys, numpy as np, pandas as pd
sys.path.insert(0, str(Path('..').resolve()))
from src.utils import load_config, set_seed, ensure_dirs
cfg=load_config('../config/config.yaml'); set_seed(cfg['seed']); ensure_dirs(cfg)
cfg

## 1. Load and preprocess corpus

In [ ]:
from src.preprocess import preprocess_dataframe
df=pd.read_csv('../data/raw/openalex_bes.csv')
df=preprocess_dataframe(df)
df.shape

## 2. SciBERT document embeddings

In [ ]:
from src.embeddings import SciBERTEmbedder
embedder=SciBERTEmbedder(cfg['scibert']['model_name'],cfg['scibert']['max_length'])
E=embedder.encode(df.text_semantic.fillna('').tolist())
np.save('../data/processed/scibert_embeddings.npy',E)
E.shape

## 3. BERTopic: 5-D UMAP + HDBSCAN + c-TF-IDF

In [ ]:
from src.topics import fit_topics, topic_table, cv_coherence, assign_topic_domains
model,topics,probs=fit_topics(df.text_semantic.fillna('').tolist(),E,cfg)
df['topic']=topics
topic_table(model).head(40)

## 4. Cᵥ coherence and T1–T4 thematic organization

In [ ]:
coh=cv_coherence(model,[x.split() for x in df.text_semantic.fillna('')],10)
domains=assign_topic_domains(model)
df['thematic_domain']=df.topic.map(domains)
coh.describe()

**Manual scientific audit required:** inspect all 37 topic terms and confirm the T1–T4 higher-order assignment before using the labels in the manuscript.

## 5. Independent 2-D UMAP for visualization

In [ ]:
from src.semantic_map import make_2d_umap
z=make_2d_umap(E,cfg)
df=pd.concat([df.reset_index(drop=True),z],axis=1)
z.head()

## 6. Linguistic indicators: LD, MATTR, HF, BF

In [ ]:
from src.linguistics import add_linguistic_features
df=add_linguistic_features(df,cfg,'../resources/hedges.txt','../resources/boosters.txt')
df[['lexical_density','mattr','hedging_frequency','boosting_frequency']].describe()

## 7. Experimental variables

In [ ]:
from src.experimental import add_experimental_features
df=add_experimental_features(df)
df[['current_density','power_density','hydrogen_efficiency','applied_potential','pH','substrate_type']].notna().sum()

## 8. Statistical analyses

In [ ]:
from src.statistics import group_test, fdr_table, spearman_with_ci
vars_=['lexical_density','mattr','hedging_frequency','boosting_frequency']
stats_table=fdr_table([group_test(df,v) for v in vars_])
stats_table

## 9. Random Forest — stratified 5-fold CV

In [ ]:
from src.ml import cross_validate_rf
pipe,metrics,cm,report,idx,num,cat=cross_validate_rf(df,cfg)
metrics, cm

## 10. SHAP and normalized mean absolute SHAP importance

In [ ]:
from src.shap_analysis import global_shap_importance, aggregate_onehot_importance
d=df.loc[idx]; X=d[num+cat]
imp=global_shap_importance(pipe,X)
agg=aggregate_onehot_importance(imp)
agg

## 11. Save empirical outputs

In [ ]:
Path('../outputs/tables').mkdir(parents=True,exist_ok=True)
df.to_csv('../data/processed/bes_processed.csv',index=False)
pd.DataFrame([metrics]).to_csv('../outputs/tables/rf_cv_metrics.csv',index=False)
cm.to_csv('../outputs/tables/rf_confusion_matrix.csv')
agg.to_csv('../outputs/tables/shap_normalized_importance.csv',index=False)